# File & Data Tools — Data Manipulation for Agents

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/16_file_and_data_tools.ipynb)

## What This Notebook Teaches You

Many agent tasks involve working with **structured data** — reading CSV files, querying JSON documents, computing statistics, and transforming datasets. Agents need tools for these operations just as they need search tools for information retrieval.

In this notebook, you will:

1. **Build a VirtualFS** — a dict-based in-memory filesystem for agents
2. **Create CSV tools** — read, query, filter, and compute statistics on tabular data
3. **Create JSON tools** — read, query, and transform nested data structures
4. **Implement statistics from scratch** — mean, median, std dev, correlation (no pandas/scipy)
5. **Build a data analysis agent** — that uses all tools to answer analytical questions

Everything is built from scratch using only Python's standard library (plus our LLM).

---

> **Prerequisites**: Notebooks 01–05 (agents, tools), Notebook 15 (search tools) recommended.
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Data-Centric Agent Tasks

Agents often need to:

- **Read files** — load data from various formats (CSV, JSON, text)
- **Query data** — filter rows, select columns, search for patterns
- **Transform data** — reshape, aggregate, join datasets
- **Compute statistics** — summarize numerical data with means, medians, correlations
- **Write results** — save transformed data, generate reports

Without data tools, agents are limited to reasoning about data described in natural language. With tools, they can **directly manipulate** datasets with precision.

### The Tool Design Pattern

Each data tool follows the same pattern we established in Notebook 05:

```python
@dataclass
class Tool:
    name: str
    description: str      # LLM reads this to decide when to use the tool
    parameters: dict       # Expected inputs
    function: Callable     # The actual implementation
```

Let's build a complete data manipulation toolkit.

## 2. VirtualFS — An In-Memory Filesystem

We create a simple dict-based filesystem so agents can create, read, write, list, and delete files without touching the real filesystem. This is **safe** (no side effects) and **observable** (we can inspect every operation).

In [ ]:
class VirtualFS:
    """Dict-based in-memory filesystem for agent use."""

    def __init__(self):
        self.files = {}  # path -> content
        self.log = []    # operation log

    def _log(self, op, path, success=True):
        self.log.append({"op": op, "path": path, "success": success, "time": time.time()})

    def create(self, path, content=""):
        """Create a new file. Fails if file already exists."""
        if path in self.files:
            self._log("create", path, False)
            return {"error": f"File already exists: {path}"}
        self.files[path] = content
        self._log("create", path)
        return {"status": "created", "path": path, "size": len(content)}

    def read(self, path):
        """Read file contents."""
        if path not in self.files:
            self._log("read", path, False)
            return {"error": f"File not found: {path}"}
        self._log("read", path)
        return {"path": path, "content": self.files[path], "size": len(self.files[path])}

    def write(self, path, content):
        """Write content to file (creates if doesn't exist, overwrites if it does)."""
        self.files[path] = content
        self._log("write", path)
        return {"status": "written", "path": path, "size": len(content)}

    def append(self, path, content):
        """Append content to existing file."""
        if path not in self.files:
            self._log("append", path, False)
            return {"error": f"File not found: {path}"}
        self.files[path] += content
        self._log("append", path)
        return {"status": "appended", "path": path, "size": len(self.files[path])}

    def delete(self, path):
        """Delete a file."""
        if path not in self.files:
            self._log("delete", path, False)
            return {"error": f"File not found: {path}"}
        del self.files[path]
        self._log("delete", path)
        return {"status": "deleted", "path": path}

    def list_files(self, prefix=""):
        """List all files, optionally filtered by prefix."""
        matches = [p for p in sorted(self.files.keys()) if p.startswith(prefix)]
        self._log("list", prefix or "/")
        return {"files": matches, "count": len(matches)}

    def exists(self, path):
        """Check if a file exists."""
        return {"exists": path in self.files, "path": path}

    def stats(self):
        """Get filesystem statistics."""
        return {
            "total_files": len(self.files),
            "total_bytes": sum(len(c) for c in self.files.values()),
            "operations": len(self.log)
        }

# Test the VirtualFS
fs = VirtualFS()

# Create some files
fs.create("/data/readme.txt", "This is a virtual filesystem for agent use.")
fs.create("/data/config.json", '{"version": 1, "debug": false}')
fs.create("/data/notes.txt", "Meeting notes: discuss Q3 results.")

# Test operations
print("List all files:", fs.list_files())
print("Read file:", fs.read("/data/readme.txt"))
print("File exists?", fs.exists("/data/config.json"))

# Append and overwrite
fs.append("/data/notes.txt", "\nAction item: prepare report by Friday.")
print("After append:", fs.read("/data/notes.txt"))

# Delete
fs.delete("/data/config.json")
print("After delete:", fs.list_files())
print("Stats:", fs.stats())

## 3. CSV Tools — Tabular Data Manipulation

CSV is the most common format for structured data. We build tools using Python's built-in `csv` module — no pandas required.

In [ ]:
import csv
import io

class CSVTools:
    """Tools for reading, querying, filtering, and analyzing CSV data."""

    @staticmethod
    def csv_read(content):
        """Parse CSV string into list of dicts (one per row)."""
        reader = csv.DictReader(io.StringIO(content))
        rows = list(reader)
        columns = reader.fieldnames or []
        return {
            "columns": columns,
            "row_count": len(rows),
            "rows": rows,
            "preview": rows[:3] if rows else []
        }

    @staticmethod
    def csv_query(rows, columns=None, limit=None):
        """Select specific columns from rows, with optional row limit."""
        if columns:
            rows = [{k: r[k] for k in columns if k in r} for r in rows]
        if limit:
            rows = rows[:limit]
        return {"rows": rows, "count": len(rows)}

    @staticmethod
    def csv_filter(rows, column, operator, value):
        """Filter rows based on a condition.

        Operators: ==, !=, >, <, >=, <=, contains
        """
        result = []
        for row in rows:
            cell = row.get(column, "")
            try:
                # Try numeric comparison
                cell_num = float(cell)
                val_num = float(value)
                if operator == "==" and cell_num == val_num: result.append(row)
                elif operator == "!=" and cell_num != val_num: result.append(row)
                elif operator == ">" and cell_num > val_num: result.append(row)
                elif operator == "<" and cell_num < val_num: result.append(row)
                elif operator == ">=" and cell_num >= val_num: result.append(row)
                elif operator == "<=" and cell_num <= val_num: result.append(row)
                elif operator == "contains" and value.lower() in cell.lower(): result.append(row)
            except (ValueError, TypeError):
                # String comparison
                if operator == "==" and cell == value: result.append(row)
                elif operator == "!=" and cell != value: result.append(row)
                elif operator == "contains" and value.lower() in cell.lower(): result.append(row)

        return {"rows": result, "count": len(result), "filter": f"{column} {operator} {value}"}

    @staticmethod
    def csv_statistics(rows, column):
        """Compute statistics for a numeric column."""
        values = []
        for row in rows:
            try:
                values.append(float(row[column]))
            except (ValueError, KeyError, TypeError):
                continue

        if not values:
            return {"error": f"No numeric values in column '{column}'"}

        n = len(values)
        sorted_vals = sorted(values)
        mean = sum(values) / n

        # Median
        if n % 2 == 0:
            median = (sorted_vals[n//2 - 1] + sorted_vals[n//2]) / 2
        else:
            median = sorted_vals[n//2]

        # Standard deviation
        variance = sum((x - mean) ** 2 for x in values) / n
        std_dev = math.sqrt(variance)

        return {
            "column": column,
            "count": n,
            "mean": round(mean, 2),
            "median": round(median, 2),
            "std_dev": round(std_dev, 2),
            "min": round(min(values), 2),
            "max": round(max(values), 2),
            "sum": round(sum(values), 2)
        }

    @staticmethod
    def csv_sort(rows, column, descending=False):
        """Sort rows by a column."""
        def sort_key(row):
            val = row.get(column, "")
            try:
                return (0, float(val))
            except (ValueError, TypeError):
                return (1, str(val))
        return {"rows": sorted(rows, key=sort_key, reverse=descending),
                "sorted_by": column, "descending": descending}

    @staticmethod
    def csv_group_by(rows, group_column, agg_column, agg_func="sum"):
        """Group rows and aggregate. Supports: sum, count, mean, min, max."""
        groups = defaultdict(list)
        for row in rows:
            key = row.get(group_column, "unknown")
            try:
                groups[key].append(float(row.get(agg_column, 0)))
            except (ValueError, TypeError):
                pass

        result = []
        for key, values in sorted(groups.items()):
            if agg_func == "sum": agg_val = sum(values)
            elif agg_func == "count": agg_val = len(values)
            elif agg_func == "mean": agg_val = sum(values) / len(values) if values else 0
            elif agg_func == "min": agg_val = min(values) if values else 0
            elif agg_func == "max": agg_val = max(values) if values else 0
            else: agg_val = sum(values)
            result.append({group_column: key, f"{agg_func}_{agg_column}": round(agg_val, 2)})

        return {"groups": result, "group_count": len(result)}

csv_tools = CSVTools()

# Quick test with sample data
sample_csv = """name,age,city,salary
Alice,30,New York,85000
Bob,25,San Francisco,92000
Charlie,35,New York,78000
Diana,28,Chicago,71000
Eve,32,San Francisco,95000"""

parsed = csv_tools.csv_read(sample_csv)
print("Columns:", parsed['columns'])
print("Row count:", parsed['row_count'])
print("Preview:", parsed['preview'])

In [ ]:
# Test CSV operations
rows = parsed['rows']

# Filter: employees in San Francisco
sf_employees = csv_tools.csv_filter(rows, "city", "==", "San Francisco")
print("San Francisco employees:", sf_employees)

# Statistics on salary
salary_stats = csv_tools.csv_statistics(rows, "salary")
print("\nSalary statistics:", json.dumps(salary_stats, indent=2))

# Sort by salary descending
sorted_rows = csv_tools.csv_sort(rows, "salary", descending=True)
print("\nSorted by salary (desc):")
for r in sorted_rows['rows']:
    print(f"  {r['name']:10s} ${r['salary']}")

# Group by city, average salary
grouped = csv_tools.csv_group_by(rows, "city", "salary", "mean")
print("\nAverage salary by city:")
for g in grouped['groups']:
    print(f"  {g['city']:20s} ${g['mean_salary']:,.0f}")

## 4. JSON Tools — Nested Data Manipulation

JSON data is often deeply nested. These tools help agents navigate, query, and transform JSON structures.

In [ ]:
class JSONTools:
    """Tools for reading, querying, and transforming JSON data."""

    @staticmethod
    def json_read(content):
        """Parse JSON string into Python object."""
        try:
            data = json.loads(content)
            dtype = type(data).__name__
            size = len(data) if isinstance(data, (list, dict)) else 1
            return {"data": data, "type": dtype, "size": size}
        except json.JSONDecodeError as e:
            return {"error": f"Invalid JSON: {e}"}

    @staticmethod
    def json_query(data, path):
        """Query nested JSON using dot-notation path.

        Examples: "users.0.name", "config.database.host", "items.*.price"
        """
        parts = path.split(".")
        current = data
        for part in parts:
            if part == "*" and isinstance(current, list):
                # Wildcard: apply rest of path to each element
                rest = ".".join(parts[parts.index(part)+1:])
                if rest:
                    return {"results": [JSONTools.json_query(item, rest)["result"]
                            for item in current if JSONTools.json_query(item, rest).get("result") is not None]}
                return {"results": current}
            elif isinstance(current, list):
                try:
                    current = current[int(part)]
                except (IndexError, ValueError):
                    return {"error": f"Invalid list index: {part}"}
            elif isinstance(current, dict):
                if part in current:
                    current = current[part]
                else:
                    return {"error": f"Key not found: {part}"}
            else:
                return {"error": f"Cannot traverse into {type(current).__name__}"}
        return {"result": current}

    @staticmethod
    def json_transform(data, operations):
        """Apply a sequence of transformations to JSON data.

        Operations: rename_key, add_field, remove_field, filter_list
        Each operation is a dict: {"op": "...", "params": {...}}
        """
        result = json.loads(json.dumps(data))  # deep copy

        for op_def in operations:
            op = op_def["op"]
            params = op_def.get("params", {})

            if op == "rename_key" and isinstance(result, dict):
                old_key, new_key = params["from"], params["to"]
                if old_key in result:
                    result[new_key] = result.pop(old_key)

            elif op == "add_field" and isinstance(result, dict):
                result[params["key"]] = params["value"]

            elif op == "remove_field" and isinstance(result, dict):
                result.pop(params["key"], None)

            elif op == "filter_list" and isinstance(result, list):
                key, value = params["key"], params["value"]
                result = [item for item in result if item.get(key) == value]

        return {"data": result}

    @staticmethod
    def json_flatten(data, prefix=""):
        """Flatten nested JSON into flat key-value pairs."""
        items = {}
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{prefix}.{k}" if prefix else k
                if isinstance(v, (dict, list)):
                    items.update(JSONTools.json_flatten(v, new_key))
                else:
                    items[new_key] = v
        elif isinstance(data, list):
            for i, v in enumerate(data):
                new_key = f"{prefix}.{i}"
                if isinstance(v, (dict, list)):
                    items.update(JSONTools.json_flatten(v, new_key))
                else:
                    items[new_key] = v
        return items

json_tools = JSONTools()

# Test with nested JSON
test_json = '''{
    "company": "TechCorp",
    "founded": 2015,
    "departments": [
        {"name": "Engineering", "headcount": 45, "budget": 2500000},
        {"name": "Marketing", "headcount": 12, "budget": 800000},
        {"name": "Sales", "headcount": 20, "budget": 1200000}
    ],
    "config": {
        "remote_policy": "hybrid",
        "offices": ["New York", "London", "Tokyo"]
    }
}'''

parsed = json_tools.json_read(test_json)
print("Parsed:", parsed['type'], "with", parsed['size'], "keys")

# Query nested paths
print("\nCompany:", json_tools.json_query(parsed['data'], "company"))
print("First dept:", json_tools.json_query(parsed['data'], "departments.0.name"))
print("All headcounts:", json_tools.json_query(parsed['data'], "departments.*.headcount"))
print("Offices:", json_tools.json_query(parsed['data'], "config.offices"))

# Flatten
flat = json_tools.json_flatten(parsed['data'])
print("\nFlattened keys:")
for k, v in flat.items():
    print(f"  {k}: {v}")

## 5. Statistics Tools — From Scratch

We implement common statistics functions using only Python's built-in math module. No pandas, numpy, or scipy.

In [ ]:
class StatisticsTools:
    """Statistical computations from scratch — no external libraries."""

    @staticmethod
    def mean(values):
        """Arithmetic mean."""
        if not values: return {"error": "Empty dataset"}
        return round(sum(values) / len(values), 4)

    @staticmethod
    def median(values):
        """Median value."""
        if not values: return {"error": "Empty dataset"}
        s = sorted(values)
        n = len(s)
        if n % 2 == 0:
            return round((s[n//2 - 1] + s[n//2]) / 2, 4)
        return round(s[n//2], 4)

    @staticmethod
    def mode(values):
        """Most frequent value(s)."""
        if not values: return {"error": "Empty dataset"}
        counts = Counter(values)
        max_count = max(counts.values())
        modes = [v for v, c in counts.items() if c == max_count]
        return {"modes": modes, "frequency": max_count}

    @staticmethod
    def std_dev(values, sample=True):
        """Standard deviation (sample by default, population if sample=False)."""
        if len(values) < 2: return {"error": "Need at least 2 values"}
        m = sum(values) / len(values)
        n = len(values)
        divisor = n - 1 if sample else n
        variance = sum((x - m) ** 2 for x in values) / divisor
        return round(math.sqrt(variance), 4)

    @staticmethod
    def percentile(values, p):
        """Compute the p-th percentile (0-100)."""
        if not values: return {"error": "Empty dataset"}
        s = sorted(values)
        k = (len(s) - 1) * (p / 100)
        f = math.floor(k)
        c = math.ceil(k)
        if f == c:
            return round(s[int(k)], 4)
        return round(s[f] * (c - k) + s[c] * (k - f), 4)

    @staticmethod
    def correlation(x_values, y_values):
        """Pearson correlation coefficient between two lists."""
        n = len(x_values)
        if n != len(y_values): return {"error": "Lists must have same length"}
        if n < 2: return {"error": "Need at least 2 data points"}

        x_mean = sum(x_values) / n
        y_mean = sum(y_values) / n

        numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(x_values, y_values))
        denom_x = math.sqrt(sum((x - x_mean) ** 2 for x in x_values))
        denom_y = math.sqrt(sum((y - y_mean) ** 2 for y in y_values))

        if denom_x == 0 or denom_y == 0:
            return {"error": "No variance in one or both variables"}

        r = numerator / (denom_x * denom_y)
        return round(r, 4)

    @staticmethod
    def linear_regression(x_values, y_values):
        """Simple linear regression: y = slope * x + intercept."""
        n = len(x_values)
        if n != len(y_values): return {"error": "Lists must have same length"}
        if n < 2: return {"error": "Need at least 2 data points"}

        x_mean = sum(x_values) / n
        y_mean = sum(y_values) / n

        numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(x_values, y_values))
        denominator = sum((x - x_mean) ** 2 for x in x_values)

        if denominator == 0:
            return {"error": "No variance in x values"}

        slope = numerator / denominator
        intercept = y_mean - slope * x_mean

        # R-squared
        y_pred = [slope * x + intercept for x in x_values]
        ss_res = sum((y - yp) ** 2 for y, yp in zip(y_values, y_pred))
        ss_tot = sum((y - y_mean) ** 2 for y in y_values)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

        return {
            "slope": round(slope, 4),
            "intercept": round(intercept, 4),
            "r_squared": round(r_squared, 4),
            "equation": f"y = {slope:.4f}x + {intercept:.4f}"
        }

    @staticmethod
    def summary(values):
        """Complete statistical summary."""
        if not values: return {"error": "Empty dataset"}
        stats = StatisticsTools
        return {
            "count": len(values),
            "mean": stats.mean(values),
            "median": stats.median(values),
            "std_dev": stats.std_dev(values) if len(values) >= 2 else "N/A",
            "min": round(min(values), 4),
            "max": round(max(values), 4),
            "range": round(max(values) - min(values), 4),
            "p25": stats.percentile(values, 25),
            "p75": stats.percentile(values, 75),
            "iqr": round(stats.percentile(values, 75) - stats.percentile(values, 25), 4)
        }

stats_tools = StatisticsTools()

# Test with sample data
data = [23, 45, 12, 67, 34, 56, 78, 89, 43, 21, 55, 38, 72, 61, 44]
print("Data:", data)
print("\nSummary:", json.dumps(stats_tools.summary(data), indent=2))

# Correlation test
x = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
y = [2.1, 4.3, 5.8, 8.2, 9.5, 12.1, 13.8, 16.2, 17.9, 20.3]
print("\nCorrelation (x, y):", stats_tools.correlation(x, y))
print("Regression:", json.dumps(stats_tools.linear_regression(x, y), indent=2))

## 6. Data Analysis Agent — Countries Dataset

Now we combine VirtualFS, CSV tools, and statistics tools into an agent that can answer analytical questions about real-world data.

### 6.1 — Create the Countries Dataset

In [ ]:
# Countries dataset: 20 countries with population, area, GDP, and continent
countries_csv = """name,population_millions,area_km2,gdp_billions_usd,continent
China,1412,9597000,17963,Asia
India,1408,3287000,3385,Asia
United States,334,9834000,25462,North America
Indonesia,275,1905000,1319,Asia
Pakistan,229,881000,377,Asia
Brazil,214,8516000,1920,South America
Nigeria,218,924000,477,Africa
Bangladesh,170,148000,460,Asia
Russia,144,17098000,2240,Europe
Mexico,128,1964000,1414,North America
Japan,125,378000,4231,Asia
Ethiopia,120,1104000,126,Africa
Egypt,104,1002000,477,Africa
Germany,83,357000,4259,Europe
France,68,641000,2783,Europe
United Kingdom,67,243000,3071,Europe
Italy,59,301000,2010,Europe
South Korea,52,100000,1665,Asia
Australia,26,7692000,1675,Oceania
Canada,39,9985000,2140,North America"""

# Load into VirtualFS and parse
fs = VirtualFS()
fs.create("/data/countries.csv", countries_csv)

# Parse the dataset
result = fs.read("/data/countries.csv")
parsed = csv_tools.csv_read(result['content'])
countries = parsed['rows']

print(f"Dataset: {parsed['row_count']} countries, {len(parsed['columns'])} columns")
print(f"Columns: {parsed['columns']}")
print(f"\nFirst 3 rows:")
for row in countries[:3]:
    print(f"  {row['name']:20s} pop={row['population_millions']:>5s}M  "
          f"area={row['area_km2']:>9s}km²  gdp=${row['gdp_billions_usd']:>6s}B  {row['continent']}")

### 6.2 — Building the Data Analysis Agent

In [ ]:
class DataAnalysisAgent:
    """Agent that uses CSV, JSON, and statistics tools to answer analytical questions."""

    def __init__(self, fs, csv_tools, json_tools, stats_tools):
        self.fs = fs
        self.csv = csv_tools
        self.json = json_tools
        self.stats = stats_tools
        self.available_tools = {
            "csv_read": {
                "description": "Parse CSV content into rows. Returns {columns, row_count, rows}.",
                "function": self.csv.csv_read
            },
            "csv_filter": {
                "description": "Filter rows where column operator value. Operators: ==, !=, >, <, >=, <=, contains.",
                "function": self.csv.csv_filter
            },
            "csv_statistics": {
                "description": "Compute statistics (mean, median, std_dev, min, max) for a numeric column.",
                "function": self.csv.csv_statistics
            },
            "csv_sort": {
                "description": "Sort rows by a column, ascending or descending.",
                "function": self.csv.csv_sort
            },
            "csv_group_by": {
                "description": "Group rows by a column and aggregate another column. Agg funcs: sum, count, mean, min, max.",
                "function": self.csv.csv_group_by
            },
            "stats_correlation": {
                "description": "Compute Pearson correlation between two numeric lists.",
                "function": self.stats.correlation
            },
            "stats_regression": {
                "description": "Compute linear regression (slope, intercept, r_squared) for two numeric lists.",
                "function": self.stats.linear_regression
            }
        }

    def _build_tool_descriptions(self):
        desc = []
        for name, info in self.available_tools.items():
            desc.append(f"- {name}: {info['description']}")
        return "\n".join(desc)

    def _execute_analysis(self, question, data_rows, columns):
        """Ask LLM to plan analysis steps, then execute them."""
        tool_desc = self._build_tool_descriptions()
        column_info = f"Columns: {columns}"
        sample_rows = data_rows[:3]

        messages = [
            {"role": "system", "content": (
                "You are a data analyst. Given a dataset and a question, plan the analysis steps. "
                "Output a JSON list of steps. Each step has: tool_name, params (as a dict), and reason.\n"
                f"Available tools:\n{tool_desc}\n\n"
                f"Dataset info:\n{column_info}\n"
                f"Sample rows: {json.dumps(sample_rows[:2])}\n\n"
                "IMPORTANT: Return ONLY the JSON array of steps. No markdown, no explanation."
            )},
            {"role": "user", "content": question}
        ]

        plan_text = generate(messages, max_new_tokens=500)

        # Parse plan
        try:
            # Extract JSON from response
            json_match = re.search(r'\[.*\]', plan_text, re.DOTALL)
            if json_match:
                plan = json.loads(json_match.group())
            else:
                plan = json.loads(plan_text)
        except json.JSONDecodeError:
            plan = []

        return plan, plan_text

    def analyze(self, question, data_rows, columns, verbose=True):
        """Answer an analytical question about the data."""
        if verbose:
            print(f"\n{'='*70}")
            print(f"QUESTION: {question}")
            print(f"{'='*70}")

        # Step 1: Get analysis plan from LLM
        plan, plan_text = self._execute_analysis(question, data_rows, columns)

        if verbose:
            print(f"\nAnalysis plan ({len(plan)} steps):")
            for i, step in enumerate(plan):
                tool = step.get('tool_name', 'unknown')
                reason = step.get('reason', '')
                print(f"  Step {i+1}: {tool} — {reason}")

        # Step 2: Execute analysis steps manually (since plans vary)
        # We do a direct analysis based on the question type
        results = {}

        # Basic statistics for all numeric columns
        numeric_cols = ['population_millions', 'area_km2', 'gdp_billions_usd']
        for col in numeric_cols:
            results[f"{col}_stats"] = self.csv.csv_statistics(data_rows, col)

        # Group by continent
        for col in numeric_cols:
            results[f"continent_{col}"] = self.csv.csv_group_by(
                data_rows, "continent", col, "sum"
            )
            results[f"continent_mean_{col}"] = self.csv.csv_group_by(
                data_rows, "continent", col, "mean"
            )

        # Correlations
        pop_vals = [float(r['population_millions']) for r in data_rows]
        gdp_vals = [float(r['gdp_billions_usd']) for r in data_rows]
        area_vals = [float(r['area_km2']) for r in data_rows]
        results['pop_gdp_correlation'] = self.stats.correlation(pop_vals, gdp_vals)
        results['area_gdp_correlation'] = self.stats.correlation(area_vals, gdp_vals)
        results['pop_gdp_regression'] = self.stats.linear_regression(pop_vals, gdp_vals)

        # GDP per capita
        gdp_per_cap = []
        for r in data_rows:
            gpc = float(r['gdp_billions_usd']) * 1000 / float(r['population_millions'])
            gdp_per_cap.append({"name": r['name'], "gdp_per_capita_k": round(gpc, 1)})
        gdp_per_cap.sort(key=lambda x: x['gdp_per_capita_k'], reverse=True)
        results['gdp_per_capita_ranked'] = gdp_per_cap

        # Population density
        density = []
        for r in data_rows:
            d = float(r['population_millions']) * 1e6 / float(r['area_km2'])
            density.append({"name": r['name'], "density_per_km2": round(d, 1)})
        density.sort(key=lambda x: x['density_per_km2'], reverse=True)
        results['density_ranked'] = density

        if verbose:
            print(f"\nComputed {len(results)} result sets")

        # Step 3: Synthesize answer with LLM
        # Select relevant results based on question
        context = json.dumps({k: v for k, v in results.items()}, indent=2, default=str)
        # Truncate if too long
        if len(context) > 3000:
            context = context[:3000] + "\n... (truncated)"

        messages = [
            {"role": "system", "content": (
                "You are a data analyst. Answer the question based on the analysis results provided. "
                "Be specific — cite numbers, country names, and comparisons. "
                "Format your answer clearly with key findings."
            )},
            {"role": "user", "content": (
                f"Question: {question}\n\n"
                f"Analysis Results:\n{context}"
            )}
        ]

        answer = generate(messages, max_new_tokens=400)

        if verbose:
            print(f"\n--- ANSWER ---")
            print(answer)

        return answer, results

# Create the agent
analysis_agent = DataAnalysisAgent(fs, csv_tools, json_tools, stats_tools)

In [ ]:
# Run analytical questions
analytical_questions = [
    "Which continent has the highest total GDP and which has the highest average GDP per country?",
    "What is the relationship between population and GDP? Are larger countries richer?",
    "Which are the top 5 countries by GDP per capita, and which continent dominates?",
    "Compare the population density of Asian countries vs European countries.",
]

for q in analytical_questions:
    analysis_agent.analyze(q, countries, parsed['columns'])
    print()

### 6.3 — Extended Analysis: Derived Metrics

In [ ]:
# Additional analysis: compute derived metrics and save results

# GDP per capita for all countries
print("GDP Per Capita Ranking (billions USD / millions people × 1000 = thousands USD):")
print("-" * 60)
for i, entry in enumerate(analysis_agent.analyze("", countries, parsed['columns'], verbose=False)[1]['gdp_per_capita_ranked']):
    bar = "█" * int(entry['gdp_per_capita_k'] / 3)
    print(f"  {i+1:2d}. {entry['name']:20s} ${entry['gdp_per_capita_k']:8.1f}k {bar}")

print("\n\nPopulation Density Ranking (people per km²):")
print("-" * 60)
results = analysis_agent.analyze("", countries, parsed['columns'], verbose=False)[1]
for i, entry in enumerate(results['density_ranked']):
    bar = "█" * min(int(entry['density_per_km2'] / 50), 40)
    print(f"  {i+1:2d}. {entry['name']:20s} {entry['density_per_km2']:8.1f}/km² {bar}")

# Correlation summary
print(f"\n\nCorrelations:")
print(f"  Population ↔ GDP:  r = {results['pop_gdp_correlation']}")
print(f"  Area ↔ GDP:        r = {results['area_gdp_correlation']}")
reg = results['pop_gdp_regression']
print(f"  Regression: {reg['equation']}")
print(f"  R² = {reg['r_squared']} ({'strong' if reg['r_squared'] > 0.5 else 'weak'} fit)")

## 7. Integrated Data Pipeline

Let's demonstrate all tools working together in a pipeline: read data → transform → analyze → save report.

In [ ]:
# Integrated pipeline: load, transform, analyze, report

# Step 1: Load data from VirtualFS
raw = fs.read("/data/countries.csv")
data = csv_tools.csv_read(raw['content'])
print(f"Step 1: Loaded {data['row_count']} rows, {len(data['columns'])} columns")

# Step 2: Filter to top 10 by GDP
sorted_data = csv_tools.csv_sort(data['rows'], 'gdp_billions_usd', descending=True)
top10 = sorted_data['rows'][:10]
print(f"Step 2: Filtered to top 10 by GDP")
for r in top10:
    print(f"  {r['name']:20s} GDP=${r['gdp_billions_usd']:>6s}B")

# Step 3: Compute statistics on top 10
print(f"\nStep 3: Statistics for top 10 economies")
for col in ['population_millions', 'gdp_billions_usd', 'area_km2']:
    stats = csv_tools.csv_statistics(top10, col)
    print(f"  {col}: mean={stats['mean']}, median={stats['median']}, std={stats['std_dev']}")

# Step 4: Generate report
report_data = {
    "title": "Top 10 Economies Analysis",
    "generated": "2025-01-15",
    "countries": [r['name'] for r in top10],
    "total_gdp_billions": sum(float(r['gdp_billions_usd']) for r in top10),
    "total_population_millions": sum(float(r['population_millions']) for r in top10),
    "gdp_stats": csv_tools.csv_statistics(top10, 'gdp_billions_usd'),
    "continent_breakdown": csv_tools.csv_group_by(top10, 'continent', 'gdp_billions_usd', 'sum')
}

# Save report to VirtualFS
report_json = json.dumps(report_data, indent=2)
fs.write("/reports/top10_analysis.json", report_json)
print(f"\nStep 4: Report saved to /reports/top10_analysis.json ({len(report_json)} bytes)")

# Verify
print(f"\nVirtualFS contents:")
for f_info in fs.list_files()['files']:
    content = fs.read(f_info)
    print(f"  {f_info} ({content['size']} bytes)")

## 8. Key Takeaways

### What We Built

| Tool | Purpose | Key Methods |
|------|---------|-------------|
| **VirtualFS** | Safe in-memory filesystem | create, read, write, list, delete |
| **CSVTools** | Tabular data manipulation | read, filter, sort, group_by, statistics |
| **JSONTools** | Nested data queries | read, query (dot-notation), transform, flatten |
| **StatisticsTools** | Numerical analysis | mean, median, std_dev, correlation, regression |
| **DataAnalysisAgent** | LLM + tools integration | plan → execute → synthesize |

### Key Insights

1. **VirtualFS provides safety** — agents can manipulate files without touching the real filesystem
2. **CSV tools enable precision** — filtering and statistics give exact answers, not LLM approximations
3. **JSON path queries** handle nested data that would be awkward to describe in natural language
4. **From-scratch statistics** show that agents don't need pandas — standard library suffices for many tasks
5. **The plan-execute-synthesize pattern** — LLM plans the analysis, tools execute it, LLM explains results
6. **Tool composition** — complex analysis emerges from combining simple, reliable tools

### Design Principles

- **Tools should be deterministic** — same input → same output (unlike LLM calls)
- **Return structured data** — dicts/lists, not formatted strings
- **Log operations** — essential for debugging agent behavior
- **Validate inputs** — return clear error messages, not exceptions

---

**Next Notebook**: [17_multi_agent_conversation.ipynb](./17_multi_agent_conversation.ipynb) — two agents talking to each other.